# Fabric Medallion Architecture - Setup & Deployment

This notebook creates the folder structure and validates the medallion architecture setup in your Fabric lakehouse.

## Step 1: Create Folder Structure

In [ ]:
# Import required libraries
from datetime import datetime
import json

# Configuration
WORKSPACE_NAME = "fabricaena"
LAKEHOUSE_NAME = "githubclaude"

# Folder paths in the lakehouse
FOLDERS = [
    "/Files/Bronze",
    "/Files/Silver",
    "/Files/Gold",
    "/Files/Documentation",
    "/Files/Scripts"
]

print(f"Creating folder structure in: {LAKEHOUSE_NAME}")
print(f"Workspace: {WORKSPACE_NAME}\n")

# Create placeholder files in each folder
for folder in FOLDERS:
    try:
        dbutils.fs.put(f"{folder}/.placeholder", "", overwrite=True)
        print(f"✓ Created: {folder}")
    except Exception as e:
        print(f"! Error creating {folder}: {str(e)}")

print("\n✓ Folder structure created successfully!")

## Step 2: Create Layer Documentation

In [ ]:
# Create README files for each layer

bronze_readme = """# Bronze Layer - Raw Data

This folder contains raw CSV files imported directly from Kaggle without any transformations.

## Files
- customer_master.csv - Customer master data
- ecommerce_sales_customer_analytics_150k.csv - Main sales dataset (150K records)
- order_items.csv - Order line items and product details
- product_catalog.csv - Product master data
- dataset_statistics.csv - Dataset summary statistics

## Purpose
- Store raw data as-is from source systems
- No transformations or cleaning applied
- Serves as audit trail of original data

## Data Quality
Data is ingested as-is. Quality issues are identified and resolved in the Silver layer.

## Next Step
Run the medallion_pipeline notebook to transform this data to the Silver layer.
"""

silver_readme = """# Silver Layer - Cleaned & Transformed Data

This folder contains cleaned, deduplicated data with business rule enforcement.

## Transformations Applied
- Data type conversions and standardization
- Null/missing value handling
- Duplicate record removal
- Date/time normalization
- Referential integrity validation
- Business rule enforcement

## Tables (Delta Format)
- customer_silver - Cleaned customer dimension
- sales_silver - Cleaned sales transactions
- order_items_silver - Cleaned order line items
- products_silver - Cleaned product master

## Purpose
- Serve as foundation for analytical models
- Ensure data quality and consistency
- Enable reliable reporting and analysis

## Quality Standards
- No NULL values in key fields
- No duplicate records
- All referenced IDs exist in dimension tables
- Data types are consistent

## Next Step
Create Gold layer analytical views using the SQL Analytical Endpoint script.
"""

gold_readme = """# Gold Layer - Business Analytics

This folder contains dimensional and fact tables optimized for business analytics.

## Dimensional Tables
- dim_customer - Customer attributes and profiles
- dim_product - Product attributes and hierarchy
- dim_date - Date/time attributes for time-series analysis

## Fact Tables
- fact_sales - Sales transactions with customer and date keys
- fact_order_items - Order line items with product keys

## Analytical Views
- customer_segmentation - RFM analysis and customer segments
- sales_by_category - Sales performance by product category
- monthly_sales_trend - Monthly revenue trends and KPIs
- top_products - Best-selling products and rankings
- daily_sales_summary - Daily business metrics
- customer_lifetime_value - CLV analysis and segmentation

## Purpose
- Optimized for reporting and dashboards
- Contains aggregations and business metrics
- Supports real-time analytics and Power BI
- Enables self-service analytics

## Next Step
Connect Power BI to this layer for visualization and dashboarding.
"""

docs_readme = """# Documentation

This folder contains dataset schema, transformation rules, and pipeline documentation.

## Files
- dataset_schema.md - Complete dataset schema documentation
- README.txt - Comprehensive transformation guide
- schema.json - Schema in JSON format

## Purpose
- Reference for data structure and content
- Transformation methodology documentation
- Data lineage and quality standards

## Usage
Refer to these files for:
- Understanding data structure
- Transformation logic details
- Data quality rules and standards
- Troubleshooting and validation
"""

# Write README files
print("Creating layer documentation...\n")

dbutils.fs.put("/Files/Bronze/README.md", bronze_readme, overwrite=True)
print("✓ Created: Bronze/README.md")

dbutils.fs.put("/Files/Silver/README.md", silver_readme, overwrite=True)
print("✓ Created: Silver/README.md")

dbutils.fs.put("/Files/Gold/README.md", gold_readme, overwrite=True)
print("✓ Created: Gold/README.md")

dbutils.fs.put("/Files/Documentation/README.md", docs_readme, overwrite=True)
print("✓ Created: Documentation/README.md")

print("\n✓ Documentation created successfully!")

## Step 3: Create Deployment Summary

In [ ]:
# Create deployment summary
summary = {
    "timestamp": datetime.now().isoformat(),
    "workspace": "fabricaena",
    "lakehouse": "githubclaude",
    "folders_created": [
        "Bronze",
        "Silver",
        "Gold",
        "Documentation",
        "Scripts"
    ],
    "architecture": "Medallion (Bronze-Silver-Gold)",
    "status": "Ready for deployment",
    "next_steps": [
        "1. Run medallion_pipeline notebook to upload Bronze layer",
        "2. Execute silver transformation for Silver layer",
        "3. Run SQL script in Analytical Endpoint for Gold layer",
        "4. Create Power BI dashboard for analytics"
    ]
}

summary_json = json.dumps(summary, indent=2)
dbutils.fs.put("/Files/deployment_summary.json", summary_json, overwrite=True)

print("✓ Created: deployment_summary.json")
print("\n" + json.dumps(summary, indent=2))

## Step 4: Verify Folder Structure

In [ ]:
# Verify folder structure
print("Verifying folder structure...\n")

try:
    files = dbutils.fs.ls("/Files")
    print("Folder Contents:")
    for file in files:
        if file.name.startswith("."):
            continue
        name = file.name.rstrip("/")
        size = file.size
        print(f"  {name:<40} {size:>15,} bytes")
    
    print("\n✓ Folder structure verified!")
except Exception as e:
    print(f"Error: {str(e)}")

## Step 5: Deployment Summary & Instructions

In [ ]:
print("="*80)
print("MEDALLION ARCHITECTURE - DEPLOYMENT COMPLETE")
print("="*80)

print(f"\nWorkspace: fabricaena")
print(f"Lakehouse: githubclaude")
print(f"Status: Ready for data pipeline execution")

print("\n" + "-"*80)
print("FOLDER STRUCTURE:")
print("-"*80)

print("""Files/
├── Bronze/              (Raw Kaggle CSV files)
│   └── README.md
├── Silver/              (Cleaned & transformed data)
│   └── README.md
├── Gold/                (Business analytics)
│   └── README.md
├── Documentation/       (Schema & guides)
│   └── README.md
└── Scripts/             (Notebooks & SQL scripts)
    └── README.md
""")

print("-"*80)
print("NEXT STEPS:")
print("-"*80)

print("""\n1. UPLOAD MEDALLION PIPELINE NOTEBOOK
   - Create new Notebook in this lakehouse
   - Name it: 'medallion_pipeline'
   - Use the Kaggle dataset download and transformation code
   - Run all cells to upload Bronze layer and transform to Silver

2. CREATE GOLD LAYER (SQL Analytical Endpoint)
   - Create SQL Analytical Endpoint
   - Execute the gold_layer_transformation.sql script
   - Creates dimensional, fact, and analytical tables

3. BUILD POWER BI DASHBOARD
   - Connect to Gold layer tables
   - Create visualizations for:
     * Sales trends and KPIs
     * Customer segmentation
     * Product performance
     * Revenue analysis

4. SCHEDULE AUTOMATED REFRESH
   - Set up notebook execution schedule
   - Configure data refresh frequency
   - Monitor data quality and transformations
""")

print("\n" + "="*80)
print(f"Setup completed at: {datetime.now().isoformat()}")
print("="*80)